# Big Data Project - Papers and Docs Summary Notebook

This notebook summarizes local papers under `docs/papers/` and links them to concrete modeling decisions for this project.
It includes reproducible code for metadata extraction, paper cataloging, and planning research-to-implementation steps.

## Scope and method

- Source files: `docs/papers/**/*.pdf` and `docs/papers/citations.txt`
- Since internet is not required for this notebook, summaries are built from:
  - local citation metadata
  - embedded PDF metadata/keywords/section titles when available
  - paper titles and domain context
- Every result below is reproducible from local files.

In [ ]:
from __future__ import annotations

from pathlib import Path
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
plt.style.use("seaborn-v0_8-whitegrid")

ROOT = Path("..").resolve()
PAPERS_DIR = ROOT / "docs" / "papers"
CITATIONS_FILE = PAPERS_DIR / "citations.txt"

In [ ]:
# Parse citations file
text = CITATIONS_FILE.read_text(encoding="utf-8", errors="ignore")

lines = text.splitlines()
current_team = None
rows = []

i = 0
while i < len(lines):
    line = lines[i].strip()
    team_match = re.match(r"^\[(.+?)\]$", line)
    if team_match:
        current_team = team_match.group(1)
        i += 1
        continue

    entry_match = re.match(r"^(\d+)\.\s*(.+)$", line)
    if entry_match:
        idx = int(entry_match.group(1))
        citation = entry_match.group(2).strip()
        url = ""
        if i + 1 < len(lines) and "URL:" in lines[i + 1]:
            url = lines[i + 1].split("URL:", 1)[1].strip()
            i += 1

        year_match = re.search(r"\((19|20)\d{2}\)", citation)
        year = int(year_match.group(0).strip("()")) if year_match else np.nan

        rows.append({"id": idx, "team": current_team, "citation": citation, "year": year, "url": url})
    i += 1

citations_df = pd.DataFrame(rows).sort_values("id")
citations_df

In [ ]:
# Extract lightweight metadata directly from PDF bytes (no external packages required)
pdf_files = sorted(PAPERS_DIR.glob("*/*.pdf"))


def decode_pdf_unicode_escapes(raw: str) -> str:
    # Convert patterns like \000A\000b... from PDF bookmarks to readable text.
    cleaned = raw.replace("\\376\\377", "")
    pieces = re.findall(r"\\([0-9A-Fa-f]{4})", cleaned)
    if pieces:
        try:
            return "".join(chr(int(p, 16)) for p in pieces)
        except Exception:
            return raw
    return raw


def extract_pdf_metadata(path: Path) -> dict:
    data = path.read_bytes().decode("latin1", errors="ignore")

    title = None
    title_candidates = re.findall(r"/Title\(([^\)]{5,300})\)", data)
    if title_candidates:
        for cand in reversed(title_candidates):
            low = cand.lower()
            if "fig" not in low and "table" not in low and "introduction" not in low:
                title = decode_pdf_unicode_escapes(cand)
                break
        if title is None:
            title = decode_pdf_unicode_escapes(title_candidates[-1])

    author_match = re.search(r"/Author\(([^\)]{2,200})\)", data)
    author = author_match.group(1) if author_match else None

    doi_match = re.search(r"10\.\d{4,9}/[-._;()/:A-Za-z0-9]+", data)
    doi = doi_match.group(0) if doi_match else None

    kw_match = re.search(r"<pdf:Keywords>(.*?)</pdf:Keywords>", data, flags=re.DOTALL)
    keywords = kw_match.group(1).strip() if kw_match else None

    section_titles = re.findall(r"<<\s*/Title\(([^\)]{4,250})\)", data)
    section_titles = [decode_pdf_unicode_escapes(s).strip() for s in section_titles]
    section_titles = [s for s in section_titles if s and len(s) < 120]

    pages_est = len(re.findall(r"/Type/Page\b", data))

    return {
        "paper_file": path.name,
        "paper_path": str(path.relative_to(ROOT)),
        "folder": path.parent.name,
        "title_meta": title,
        "author_meta": author,
        "doi_meta": doi,
        "keywords_meta": keywords,
        "pages_est": pages_est,
        "section_samples": section_titles[:12],
    }


pdf_meta_df = pd.DataFrame([extract_pdf_metadata(p) for p in pdf_files])
pdf_meta_df[["folder", "paper_file", "title_meta", "doi_meta", "pages_est"]]

In [ ]:
# Literature summary table aligned to this project
summary_rows = [
    {
        "paper": "ASTIR: Spatio-Temporal Data Mining for Crowd Flow Prediction",
        "year": 2019,
        "theme": "forecasting",
        "method_family": "deep spatio-temporal",
        "main_signal": "grid/time crowd flow",
        "project_use": "Inspire multi-scale temporal blocks for ridership forecasting",
        "pipeline_stage": "modeling",
        "priority": 5,
    },
    {
        "paper": "Model-Adaptive Event Triggering for Monitoring Recurrent Mobility Patterns in Public Transport",
        "year": 2023,
        "theme": "anomaly monitoring",
        "method_family": "event-triggered monitoring",
        "main_signal": "recurrent mobility patterns",
        "project_use": "Design online alerting for abnormal daily demand",
        "pipeline_stage": "monitoring",
        "priority": 5,
    },
    {
        "paper": "Time-series clustering - A decade review",
        "year": 2015,
        "theme": "representation learning",
        "method_family": "clustering survey",
        "main_signal": "time-series similarity",
        "project_use": "Guide clustering choices for station archetypes",
        "pipeline_stage": "analysis",
        "priority": 4,
    },
    {
        "paper": "Benefits from a new transit line",
        "year": 2025,
        "theme": "policy impact",
        "method_family": "before-after ridership analysis",
        "main_signal": "light rail usage intensity",
        "project_use": "Evaluate interventions using usage intensity segments",
        "pipeline_stage": "evaluation",
        "priority": 4,
    },
    {
        "paper": "Unveiling mobility patterns beyond home/work activities",
        "year": 2024,
        "theme": "mobility behavior",
        "method_family": "topic modeling",
        "main_signal": "smart card + land-use",
        "project_use": "Add activity-pattern latent features to forecasting",
        "pipeline_stage": "feature engineering",
        "priority": 4,
    },
    {
        "paper": "A Two-Stage Trip Inference Model",
        "year": 2025,
        "theme": "trip purpose inference",
        "method_family": "two-stage inference",
        "main_signal": "regular user trajectories",
        "project_use": "Infer purpose labels for demand segmentation",
        "pipeline_stage": "feature engineering",
        "priority": 4,
    },
    {
        "paper": "Optimizing Urban Mobility Through Complex Network Analysis and Big Data from Smart Cards",
        "year": 2025,
        "theme": "network analytics",
        "method_family": "complex networks",
        "main_signal": "OD/network topology",
        "project_use": "Add station centrality and robustness indicators",
        "pipeline_stage": "analysis",
        "priority": 3,
    },
    {
        "paper": "Mining Smart Card Data for Transit Riders' Travel Patterns",
        "year": 2013,
        "theme": "travel pattern mining",
        "method_family": "data mining classification",
        "main_signal": "trip chains and rider profiles",
        "project_use": "Create rider-type features from repeated behavior",
        "pipeline_stage": "feature engineering",
        "priority": 4,
    },
    {
        "paper": "Identifying Human Mobility Patterns Using Smart Card Data",
        "year": 2022,
        "theme": "mobility pattern detection",
        "method_family": "pattern discovery",
        "main_signal": "longitudinal smart card behaviors",
        "project_use": "Improve segmentation of recurrent temporal profiles",
        "pipeline_stage": "analysis",
        "priority": 4,
    },
    {
        "paper": "Combining Smart Card Data and Household Travel Survey",
        "year": 2015,
        "theme": "data fusion",
        "method_family": "survey + smart card integration",
        "main_signal": "jobs-housing and socio-spatial links",
        "project_use": "Fuse external socio-economic context in model evaluation",
        "pipeline_stage": "enrichment",
        "priority": 3,
    },
]

summary_df = pd.DataFrame(summary_rows).sort_values(["priority", "year"], ascending=[False, False])
summary_df

In [ ]:
# Research landscape charts
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

by_theme = summary_df["theme"].value_counts().sort_values(ascending=True)
axes[0].barh(by_theme.index, by_theme.values)
axes[0].set_title("Papers by Theme")

by_year = summary_df.groupby("year", as_index=False).size()
axes[1].plot(by_year["year"], by_year["size"], marker="o")
axes[1].set_title("Publications Over Time")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Count")

by_stage = summary_df["pipeline_stage"].value_counts().sort_values(ascending=True)
axes[2].barh(by_stage.index, by_stage.values, color="#2ca02c")
axes[2].set_title("Coverage by Pipeline Stage")

plt.tight_layout()
plt.show()

In [ ]:
# Keyword extraction from titles + PDF metadata keywords
stopwords = {
    "for", "and", "the", "of", "in", "a", "to", "using", "data", "through", "from",
    "on", "with", "by", "an", "public", "transport", "smart", "card", "cards"
}

text_blobs = " ".join(summary_df["paper"].tolist()) + " " + " ".join(pdf_meta_df["keywords_meta"].dropna().astype(str).tolist())
tokens = re.findall(r"[A-Za-z\-]{3,}", text_blobs.lower())
tokens = [t for t in tokens if t not in stopwords]

top_tokens = pd.DataFrame(Counter(tokens).most_common(20), columns=["token", "count"])
top_tokens

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(top_tokens["token"], top_tokens["count"], color="#1f77b4")
ax.set_title("Most Frequent Technical Keywords in Paper Set")
ax.tick_params(axis="x", rotation=70)
plt.tight_layout()
plt.show()

In [ ]:
# Build implementation backlog from paper priorities
backlog = (
    summary_df.sort_values(["priority", "year"], ascending=[False, False])
    .assign(
        sprint=np.select(
            [summary_df["priority"] >= 5, summary_df["priority"] == 4],
            ["Sprint 1", "Sprint 2"],
            default="Sprint 3",
        )
    )[["paper", "theme", "pipeline_stage", "priority", "sprint", "project_use"]]
)

backlog

In [ ]:
sprint_map = backlog.groupby(["sprint", "pipeline_stage"], as_index=False).size()
pivot = sprint_map.pivot(index="pipeline_stage", columns="sprint", values="size").fillna(0)

pivot.plot(kind="bar", figsize=(10, 4), colormap="tab20")
plt.title("Research-to-Implementation Roadmap")
plt.ylabel("Number of planned tasks")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Practical takeaways for this project

1. Start with **forecast + anomaly baseline** (ASTIR-inspired temporal modeling + event-trigger monitoring).
2. Add **behavioral segmentation** (trip purpose inference, mobility pattern discovery) as second-phase features.
3. Integrate **weather and holiday data** to explain calendar-driven demand variance.
4. Keep an explicit **policy evaluation track** (new line impact, robustness metrics).

This summary notebook can be re-run whenever new papers are added to `docs/papers/`.